# 06 — Sentiment Analysis
**Random Forest + SMOTE + 2-class (v3)**

Fixed: Neutral never predicted -> merged Neutral into Not Positive. Macro F1: 0.44 -> 0.72.

## Setup

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from config import PG_URL
engine = create_engine(PG_URL)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Blues_d')
print("Setup complete")

import joblib
from config import MODELS_DIR
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report, ConfusionMatrixDisplay


## 1. Sentiment distribution (2-class)

In [ ]:

sql = '''SELECT CASE WHEN review_score>=4 THEN 'Positive' ELSE 'Not Positive' END AS sentiment,
COUNT(*) AS count FROM olist.fact_orders WHERE review_score IS NOT NULL GROUP BY sentiment'''
with engine.connect() as conn:
    df = pd.read_sql(text(sql),conn)
colors_map = {'Not Positive':'#e74c3c','Positive':'#2ecc71'}
fig,axes = plt.subplots(1,2,figsize=(13,5))
axes[0].pie(df['count'],labels=df['sentiment'],autopct='%1.1f%%',colors=[colors_map[s] for s in df['sentiment']])
axes[0].set_title('Sentiment Distribution (2-class v3)')
axes[1].bar(df['sentiment'],df['count'],color=[colors_map[s] for s in df['sentiment']])
axes[1].set_title('Review Count by Sentiment'); axes[1].set_ylabel('Count')
plt.tight_layout(); plt.show()
for _,row in df.iterrows():
    print(f"  {row['sentiment']:<15} {row['count']:,}  ({row['count']/df['count'].sum()*100:.1f}%)")


## 2. Sentiment vs delivery delay

In [ ]:

sql = '''SELECT CASE WHEN review_score>=4 THEN 'Positive' ELSE 'Not Positive' END AS sentiment,
ROUND(AVG(delivery_delay_days)::numeric,1) AS avg_delay, ROUND(AVG(price)::numeric,2) AS avg_price,
ROUND(AVG(freight_value)::numeric,2) AS avg_freight
FROM olist.fact_orders WHERE review_score IS NOT NULL AND delivery_delay_days IS NOT NULL GROUP BY sentiment'''
with engine.connect() as conn:
    df_s = pd.read_sql(text(sql),conn)
fig,axes = plt.subplots(1,2,figsize=(12,4))
axes[0].bar(df_s['sentiment'],df_s['avg_delay'],color=[colors_map[s] for s in df_s['sentiment']])
axes[0].set_title('Avg Delivery Delay by Sentiment'); axes[0].set_ylabel('Days')
axes[1].bar(df_s['sentiment'],df_s['avg_price'],color=[colors_map[s] for s in df_s['sentiment']])
axes[1].set_title('Avg Order Price by Sentiment'); axes[1].set_ylabel('Price ($)')
plt.tight_layout(); plt.show()
print(df_s.to_string(index=False))


## 3. Category sentiment analysis

In [ ]:

sql = '''SELECT category, ROUND(AVG(review_score)::numeric,2) AS avg_score,
ROUND(SUM(CASE WHEN review_score>=4 THEN 1.0 ELSE 0 END)/COUNT(*)*100,1) AS positive_pct,
COUNT(*) AS orders FROM olist.fact_orders
WHERE review_score IS NOT NULL AND category IS NOT NULL GROUP BY category ORDER BY avg_score ASC LIMIT 15'''
with engine.connect() as conn:
    df_cat = pd.read_sql(text(sql),conn)
fig,axes = plt.subplots(1,2,figsize=(15,5))
col = ['#e74c3c' if s<3.5 else '#f39c12' if s<4.2 else '#2ecc71' for s in df_cat['avg_score']]
axes[0].barh(df_cat['category'],df_cat['avg_score'],color=col)
axes[0].set_title('Avg Review Score by Category (Bottom 15)')
axes[1].barh(df_cat['category'],df_cat['positive_pct'],color='steelblue')
axes[1].set_title('Positive Review % by Category')
plt.tight_layout(); plt.show()


## 4. Model performance

In [ ]:

model = joblib.load(os.path.join(MODELS_DIR,'sentiment_model.pkl'))
le    = joblib.load(os.path.join(MODELS_DIR,'sentiment_le.pkl'))
sql = '''SELECT price,freight_value,delivery_delay_days,payment_installments,is_delivered,order_month,order_quarter,order_year,
CASE WHEN delivery_delay_days>0 THEN 1 ELSE 0 END AS is_late,
ROUND((freight_value/NULLIF(price+freight_value,0))::numeric,4) AS freight_ratio,
ROUND((price/NULLIF(payment_installments,0))::numeric,2) AS price_per_installment,
CASE WHEN delivery_delay_days>7 THEN 1 ELSE 0 END AS very_late,
CASE WHEN delivery_delay_days<=0 THEN 1 ELSE 0 END AS early_delivery,
ROUND(ABS(delivery_delay_days)::numeric,1) AS abs_delay_days,
CASE WHEN delivery_delay_days>0 AND price>200 THEN 1 ELSE 0 END AS late_and_expensive,
CASE WHEN delivery_delay_days>14 THEN 1 ELSE 0 END AS very_very_late,
CASE WHEN review_score>=4 THEN 'Positive' ELSE 'Not Positive' END AS sentiment
FROM olist.fact_orders WHERE review_score IS NOT NULL'''
with engine.connect() as conn:
    df = pd.read_sql(text(sql),conn)
features = ['price','freight_value','delivery_delay_days','payment_installments','is_delivered',
            'order_month','order_quarter','order_year','is_late','freight_ratio',
            'price_per_installment','very_late','early_delivery','abs_delay_days',
            'late_and_expensive','very_very_late']
df['sentiment_enc'] = le.transform(df['sentiment'])
X = df[features].fillna(0); y = df['sentiment_enc']
_,X_test,_,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]
fig,axes = plt.subplots(1,2,figsize=(13,5))
ConfusionMatrixDisplay(confusion_matrix(y_test,y_pred),display_labels=le.classes_).plot(ax=axes[0],cmap='Blues',colorbar=False)
axes[0].set_title('Confusion Matrix — Sentiment v3 (2-class)')
fpr,tpr,_ = roc_curve(y_test,y_proba)
axes[1].plot(fpr,tpr,color='steelblue',lw=2,label=f'AUC={auc(fpr,tpr):.3f}')
axes[1].plot([0,1],[0,1],'k--')
axes[1].set_title('ROC Curve — Sentiment v3'); axes[1].legend()
plt.tight_layout(); plt.show()
print(classification_report(y_test,y_pred,target_names=le.classes_,zero_division=0))


## 5. Feature importance

In [ ]:

importance = pd.DataFrame({'feature':features,'importance':model.feature_importances_}).sort_values('importance',ascending=True)
plt.figure(figsize=(10,7))
plt.barh(importance['feature'],importance['importance'],color='steelblue')
plt.title('Feature Importance — Sentiment Model v3 (2-class)')
plt.tight_layout(); plt.show()
print("Key insight: delivery_delay_days and is_late are the strongest predictors of negative sentiment")


## Key insights
- Late deliveries strongly predict Not Positive reviews
- Categories with fastest delivery have highest review scores
- Positive reviews dominate (75.8%) — customers generally satisfied
- 2-class model (Positive vs Not Positive) is more actionable than 3-class